<a href="https://colab.research.google.com/github/rafiebazine/rag-image-captioning/blob/main/phase_01_data_exploration/download_and_extract_raw_coco_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Downloading and Extracting COCO 2017 Dataset
This notebook downloads the annotations, training dataset, and validation dataset from COCO 2017, and then extracts them.

# Download The data set + Unzip it.

In [ ]:
# 1. Download annotations, train images, and validation images
!wget -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!wget -q --show-progress http://images.cocodataset.org/zips/train2017.zip
!wget -q --show-progress http://images.cocodataset.org/zips/val2017.zip

annotations_trainva 100%[===================>] 241.19M  20.9MB/s    in 13s     
train2017.zip       100%[===================>]  18.01G  20.5MB/s    in 15m 23s 
val2017.zip         100%[===================>] 777.80M  21.3MB/s    in 38s     


In [ ]:
# 2. Extract all downloaded zip files
print("Extracting annotations...")
!unzip -q annotations_trainval2017.zip

Extracting annotations...


In [ ]:
print("Extracting training dataset (this may take a while)...")
!unzip -q train2017.zip

Extracting training dataset (this may take a while)...


In [ ]:
print("Extracting validation dataset...")
!unzip -q val2017.zip

Extracting validation dataset...


# Extract Sub-sets

## Extract Test subset
`100 images + 500 captions`

In [ ]:
import json
import os
import random

# Fixed seed for exact reproducibility
random.seed(42)

# File paths for Google Colab environment
CAPTIONS_FILE = os.path.join("annotations", "captions_val2017.json")
INSTANCES_FILE = os.path.join("annotations", "instances_val2017.json")
IMAGES_DIR = "val2017"
OUTPUT_DIR = "splits"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "test_100.json")

# Category grouping rules
INDOOR_GROUPS = {"furniture", "kitchen", "indoor", "appliance", "electronic"}
OUTDOOR_GROUPS = {"outdoor", "vehicle", "animal", "sports"}


def read_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)


def map_category_contexts(instances_data):
    category_map = {}
    for cat in instances_data["categories"]:
        cat_id = cat["id"]
        group = cat["supercategory"].lower()

        if group in INDOOR_GROUPS:
            category_map[cat_id] = "indoor"
        elif group in OUTDOOR_GROUPS:
            category_map[cat_id] = "outdoor"
        else:
            category_map[cat_id] = "other"

    return category_map


def build_test_dataset():
    # Load annotations
    captions_data = read_json(CAPTIONS_FILE)
    instances_data = read_json(INSTANCES_FILE)
    category_map = map_category_contexts(instances_data)

    # 1. Step 1: Filter candidates with >= 5 captions that physically exist on disk
    # Working with captions_data

    # Group images captions by image id
    captions_by_image = {}
    for ann in captions_data["annotations"]:
        captions_by_image.setdefault(ann["image_id"], []).append(ann["caption"])

    # Create a dictionary {img_id : img} to avoid repeated searches later
    images_by_id = {img["id"]: img for img in captions_data["images"]}

    # Build candidates list (>= 5 captions + exists in images list + exists in disc)
    # We take exactly 5 captions for each candidate image.
    candidates = {}
    for img_id, captions in captions_by_image.items():
        if len(captions) >= 5 and img_id in images_by_id:
            file_name = images_by_id[img_id]["file_name"]
            image_path = os.path.join(IMAGES_DIR, file_name)

            if os.path.exists(image_path):
                candidates[img_id] = {
                    "image_id": img_id,
                    "file_name": file_name,
                    "height": images_by_id[img_id].get("height"),
                    "width": images_by_id[img_id].get("width"),
                    "captions": captions[:5],  # Truncate to exactly 5 captions
                }

    print(f"Step 1 Complete: {len(candidates)} valid candidate images found.")


    # -----------------------------------------
    # Wroking with instances_data
    #
    # 2. Extract object instances per candidate
    # Group candidate images instances by img_id {img_id : instances}
    # It will be used later to filter categories

    instances_by_image = {}
    for ann in instances_data["annotations"]:
        img_id = ann["image_id"]
        if img_id in candidates:
            instances_by_image.setdefault(img_id, []).append(ann["category_id"])

    # 3. Step 2: Stratify candidates into 8 buckets & extract exactly 100 test images.
    # 8 buckets: simple-indoor-no_interraction / simple-outdoor-no_interraction / ....
    buckets = {i: [] for i in range(1, 9)}

    for img_id, item in candidates.items():
        cat_ids = instances_by_image.get(img_id, [])
        obj_count = len(cat_ids)
        distinct_count = len(set(cat_ids))

        # Complexity: Simple (1-2 objects) vs Complex (>= 3 objects)
        # skip when image contians no object at all
        is_simple = 1 <= obj_count <= 2
        is_complex = obj_count >= 3
        if not (is_simple or is_complex):
            continue

        # Interaction: No Interaction (1 category) vs Multi-Object Interaction (>= 2 categories)
        has_interaction = distinct_count >= 2

        # Context: Majority vote between indoor/outdoor categories
        # For each category id in the image, we get the id of the category
        # if the id of the category doesn't exist in our list, we put other.

        contexts = [category_map.get(cid, "other") for cid in cat_ids]
        indoor_cnt = contexts.count("indoor")
        outdoor_cnt = contexts.count("outdoor")
        # define context of the image
        context = "indoor" if indoor_cnt > outdoor_cnt else "outdoor"

        # Bucket assignment (1 to 8)
        if is_simple and not has_interaction and context == "indoor":
            bucket = 1
        elif is_simple and not has_interaction and context == "outdoor":
            bucket = 2
        elif is_simple and has_interaction and context == "indoor":
            bucket = 3
        elif is_simple and has_interaction and context == "outdoor":
            bucket = 4
        elif is_complex and not has_interaction and context == "indoor":
            bucket = 5
        elif is_complex and not has_interaction and context == "outdoor":
            bucket = 6
        elif is_complex and has_interaction and context == "indoor":
            bucket = 7
        elif is_complex and has_interaction and context == "outdoor":
            bucket = 8
        else:
            continue

        # Add instance information to the image and append it to its
        # adequate bucket.
        record = item.copy()
        record["stratified_category"] = bucket
        record["instance_count"] = obj_count
        record["distinct_categories"] = distinct_count
        record["context"] = context
        buckets[bucket].append(record)

    print("Candidate counts per bucket:")
    for b_id, items in buckets.items():
        print(f"  Bucket {b_id}: {len(items)} candidates")

    # 4. Deterministic sampling: Buckets 1-4 get 13 images, Buckets 5-8 get 12 images (Total = 100)
    bucket_targets = {1: 13, 2: 13, 3: 13, 4: 13, 5: 12, 6: 12, 7: 12, 8: 12}
    test_set = []

    for bucket, target in bucket_targets.items():
        available = buckets[bucket]
        if len(available) < target:
            raise ValueError(
                f"Bucket {bucket} has insufficient images: requested {target}, found {len(available)}"
            )

        sampled = random.sample(available, target)
        test_set.extend(sampled)

    # Sanity checks
    # Verify that the final results are correct
    assert len(test_set) == 100, f"Expected 100 images, got {len(test_set)}"
    assert sum(len(x["captions"]) for x in test_set) == 500, "Expected 500 total captions! but got less!"
    assert all(len(x["captions"]) == 5 for x in test_set), "An image does not have exactly 5 captions!"

    # Save to splits/test_100.json
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(OUTPUT_FILE, "w") as f:
        json.dump(test_set, f, indent=2)

    print(f"\n✅ Test set created: {len(test_set)} images & 500 captions saved to {OUTPUT_FILE}")


if __name__ == "__main__":
    build_test_dataset()

Step 1 Complete: 5000 valid candidate images found.
Candidate counts per bucket:
  Bucket 1: 206 candidates
  Bucket 2: 590 candidates
  Bucket 3: 190 candidates
  Bucket 4: 532 candidates
  Bucket 5: 19 candidates
  Bucket 6: 210 candidates
  Bucket 7: 1483 candidates
  Bucket 8: 1722 candidates

✅ Test set created: 100 images & 500 captions saved to splits/test_100.json


## Extract RAG knowledge base & validation subset
`RAG Knowledge Base: 10k images + 50k captions`
`Validation subset: 100 images + 500 captions`


In [8]:
import json
import os

# File paths for Google Colab environment
TEST_FILE = os.path.join("splits", "test_100.json")
CAPTIONS_FILE = os.path.join("annotations", "captions_train2017.json")
IMAGES_DIR = "train2017"
OUTPUT_DIR = "splits"
KB_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "kb_10k.json")
VAL_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "val_100.json")


def read_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)


def build_kb_and_val_datasets():
    # 1. Load reserved test image IDs to prevent leakage
    test_data = read_json(TEST_FILE)

    # build a set of test images ids
    test_ids = {item["image_id"] for item in test_data}
    print(f"Loaded {len(test_ids)} reserved test image IDs for exclusion.")

    # 2. Load train2017 captions
    captions_data = read_json(CAPTIONS_FILE)

    # Group captions by image_id
    captions_by_image = {}
    for ann in captions_data["annotations"]:
        captions_by_image.setdefault(ann["image_id"], []).append(ann["caption"])

    # Create images dictionary: {img_id : img}
    images_by_id = {img["id"]: img for img in captions_data["images"]}

    # Dictionary maps for unique image storage by image_id
    kb_10k = {}
    val_100 = {}

    # 3. Single-pass loop through candidate images
    for img_id, captions in captions_by_image.items():

        # Stop immediately when both targets are reached
        if len(kb_10k) == 10000 and len(val_100) == 100:
            break

        # Check 1: Exclude reserved test set images
        if img_id in test_ids:
            continue

        # Check 2: Exclude if fewer than 5 captions available Or
        # the image doesn't in the images dictionary
        if len(captions) < 5 or img_id not in images_by_id:
            continue

        # Check 3: Disk existence check
        file_name = images_by_id[img_id]["file_name"]
        image_path = os.path.join(IMAGES_DIR, file_name)

        # Skip when image deosn't exist phisically on the disk
        if not os.path.exists(image_path):
            continue

        # Build candidate object with exactly 5 captions
        record = {
            "image_id": img_id,
            "file_name": file_name,
            "height": images_by_id[img_id].get("height"),
            "width": images_by_id[img_id].get("width"),
            "captions": captions[:5],  # Truncate to first 5 captions
        }

        # Phase A: Populate kb_10k dictionary up to 10,000 entries
        if len(kb_10k) < 10000:
            kb_10k[img_id] = record

        # Phase B: Populate val_100 dictionary up to 100 entries
        elif len(val_100) < 100:
            # Check 4: Confirm image is not in kb_10k or val_100
            if img_id not in kb_10k and img_id not in val_100:
                val_100[img_id] = record

    print("\nExtraction Summary:")
    print(f"  KB 10k size: {len(kb_10k)} images (50,000 captions)")
    print(f"  Val 100 size: {len(val_100)} images (500 captions)")

    # 4. Strict assertion audits

    # Guarantee the the exact subset lengthes
    assert len(kb_10k) == 10000, f"Expected 10,000 KB images, got {len(kb_10k)}"
    assert len(val_100) == 100, f"Expected 100 Val images, got {len(val_100)}"
    # Gurantee No overlap between sub-datasets
    assert set(kb_10k.keys()).isdisjoint(set(val_100.keys())), "Overlap detected between KB and Val!"
    assert set(kb_10k.keys()).isdisjoint(test_ids), "Test image leaked into KB set!"
    assert set(val_100.keys()).isdisjoint(test_ids), "Test image leaked into Val set!"

    # 5. Export JSON split files
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(KB_OUTPUT_FILE, "w") as f:
        json.dump(list(kb_10k.values()), f, indent=2)

    with open(VAL_OUTPUT_FILE, "w") as f:
        json.dump(list(val_100.values()), f, indent=2)

    print(f"\n✅ Splits successfully created and saved:")
    print(f"   - {KB_OUTPUT_FILE}")
    print(f"   - {VAL_OUTPUT_FILE}")


if __name__ == "__main__":
    build_kb_and_val_datasets()

Loaded 100 reserved test image IDs for exclusion.

Extraction Summary:
  KB 10k size: 10000 images (50,000 captions)
  Val 100 size: 100 images (500 captions)

✅ Splits successfully created and saved:
   - splits/kb_10k.json
   - splits/val_100.json
